# PC-MP-PINN training, validation and testing
This notebook reproduces the project pipeline. It keeps the publication protocol and shorter quick-validation protocol separate.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.paper.dataset import generate_dataset, load_config
from src.paper.experiment import run_matrix
CONFIG = ROOT / 'configs/paper_experiment.yaml'
config = load_config(CONFIG)
config['project']

## 1. Generate or load the reference dataset
PyBaMM SPMe with lumped thermal physics and Chen2020 parameters produces 18 condition-level simulations. Initial cell temperature is set to ambient temperature.

In [ ]:
dataset = generate_dataset(CONFIG, force=False)
print(dataset.shape)
dataset.groupby(['split','condition_id']).size().groupby(level=0).agg(['count','sum'])

In [ ]:
dataset[['c_rate','ambient_C','time_s','radius_star','concentration_mol_m3','temperature_K','sigma_tangential_MPa']].describe().T

## 2. Train the comparison matrix
Quick mode runs 60 epochs as a pipeline verification. Set `FULL_PROTOCOL = True` for the paper's 2,500-epoch maximum, patience 350, and complete collocation budget.

In [ ]:
FULL_PROTOCOL = False
metrics = run_matrix(CONFIG, quick=not FULL_PROTOCOL)
metrics.head()

## 3. Validation and held-out test metrics
Nine metrics cover concentration, surface behaviour, temperature rise, particle stress, mass conservation, concentration-stress consistency and speed.

In [ ]:
summary = (metrics.groupby(['mode','split','variant','label_budget','metric']).value
           .agg(['median','mean','std']).reset_index())
summary[summary.split.eq('test')].pivot_table(index=['variant','label_budget'], columns='metric', values='median').round(4)

In [ ]:
import matplotlib.pyplot as plt
plot = summary[(summary.split == 'test') & (summary.metric == 'concentration_relative_l2')]
for model, group in plot.groupby('variant'):
    plt.plot(group.label_budget, group['median'], marker='o', label=model)
plt.xlabel('Labels per training condition'); plt.ylabel('Concentration relative L2'); plt.legend(); plt.grid(alpha=.25);

## Interpretation
The proposed model derives stress from its predicted concentration profile. Its concentration-stress consistency error should remain at numerical precision. Quick-mode values validate code and artifact flow; only full-protocol runs should be used in the paper's final result table.